In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
#from numba import jit
import random

In [ ]:
##################################################
n =2.5540
tau = 60             #60-min doubling time
k_tr = 2.73       
K = 11.35/1.26  
s0 =48.4144/(1.4*n)        #4.62500352/n #WT   #13.58594784/n  #rom-1
##################################################

x0=20          #number of plasmids in the initial cell
max_size=500   #total number of possible clusters, can be increased to accomodate larger sims
Nmax =2**10    #total number of simulated cells

########################### modify this for simulations ###############################
beta = 0.25 #plasmid recruitment rate constant
gamma = 0.05 #plasmid combination rate constant
sigma= 0.08  #plasmid dissociation rate constant
rho = 0.1  #plasmid transfer rate constant



In [ ]:
class cell:
    def __init__(self,index,init_clusters,parent,birth,tau):
        '''Initializes cell attributes
        param:: index: cell index
        param:: clusters: count of n-clusters
        param:: x: plasmid trajectory in time 
        param:: t: time stamps for x
        param:: parent: parent cell
        param:: birth: birth time
        return:: None
        '''
        self.index=index
        self.init_clusters=init_clusters
        self.fin_clusters=[]
        self.x = []
        self.t = []
        self.parent = parent
        self.birth = birth
        self.div = birth+tau+np.random.normal(0,1)

#@jit
def find_tot_plasmids(y1,y2,max_size):
    y1[0]+=y1[1]   #because of plasmid shedding y[1] may have a nonzero value, we transfer this to y[0].
    y2[0]+=y2[1]
    y1[1]=y1[0]
    y2[1]=y2[0]
    y_tot1 = np.dot(np.arange(1,max_size),y1[1:max_size])
    y_tot2 = np.dot(np.arange(1,max_size),y2[1:max_size])            
    y_tot = y_tot1+y_tot2
    y1[1]=0
    y2[1]=0
    return y1,y2,y_tot1,y_tot2,y_tot 

#@jit
def pop_check(cells,T):
    b_filter = [x for x in cells if x.t[0] <= T]
    d_filter = [x for x in b_filter if x.t[-1] >T]
    return d_filter #N_pop

#@jit
def hazards(y,c_idx,num_reax,N,y_totp,y_tot,beta,gamma,sigma,max_size):
    a =np.zeros(num_reax)                                     #reaction hazards 
    v=np.zeros([num_reax,max_size])                           #reaction vectors    
    a[0]=(k_tr*y_totp)/(1+K*y_tot/(s0*n))**n
    v[0,0] = 1
    a[1] = 0.5*beta*y[0]*(y[0]-1)
    v[1,0] = -2
    v[1,2] = 1

    if N>1:
        species_idx =np.zeros([2,num_reax])
        count=1
        for i in np.arange(1,N):   #plasmid recruitment
            count+=1
            a[count] = beta*y[0]*y[c_idx[i]]
            species_idx[:,count] = np.array([c_idx[i],0])
        rec_count=count
                
        for i in np.arange(1,N):   #cluster combination
            for j in np.arange(i,N):
                count+=1
                if i==j:
                    a[count] = 0.5*gamma*y[c_idx[i]]*(y[c_idx[j]]-1)
                else:
                    a[count] = gamma*y[c_idx[i]]*y[c_idx[j]]
                species_idx[:,count] = [c_idx[i],c_idx[j]]
        cluster_count = count
            
        for i in np.arange(1,N): #plasmid shedding
            count+=1
            a[count] = sigma*y[c_idx[i]]*c_idx[i]
            species_idx[:,count] = np.array([c_idx[i],0])
                    
        for i in range(len(a)): #reaction vectors
            if i==0:                         #plasmid replication
                v[i,0] = 1
            elif i==1:                       #plasmid duplex formation
                v[i,0] = -2
                v[i,2] = 1
            elif (2<=i and i<=rec_count):    #plasmid recruitment to a cluster
                v[i,0] = -1
                v[i,int(species_idx[0,i])] = -1
                v[i,int(species_idx[0,i])+1] = 1
            elif (i>rec_count and i<=cluster_count):   #cluster combination
                if (int(species_idx[0,i])==int(species_idx[1,i])):
                    v[i,int(species_idx[0,i])] = -2
                else:
                    v[i,int(species_idx[0,i])] = -1
                    v[i,int(species_idx[1,i])] = -1
                v[i,int(np.sum(species_idx[:,i]))] = 1
            else:                            #plasmid shedding         
                v[i,0] = 1
                v[i,int(species_idx[0,i])] = -1
                v[i,int(species_idx[0,i])-1] = 1   
    return a,v


#@jit
def ssa(y0,x0,tau,beta,gamma,sigma,max_size):
    y=y0               #plasmid count count; y[0]=x[0], y[1]=0, y[i] = number of cluster of size i     
    x=[x0]
    t_x = [0]
    t_y=0             #reaction time 
    y_tot1=x0
    y_tot2 = 0
    y1 = y[:max_size]
    y2 = y[max_size:2*max_size] 
    y_tot=y_tot1+y_tot2
    while t_y<tau:
        ############################### pole 1 ##################################
        dummy = y1[0]                                               
        y1[0] = 1
        c_idx1 = np.where(y1>0)[0]                                  #indices of species with non-zero state except possibly free plasmids
        y1[0] = dummy
        N1 = len(c_idx1)                                            #number of non-zero species
        num_reax1 = N1+1+int(((N1-1)**2-N1+1)/2)+2*(N1-1)           #possible number of reactions given the system state 
        a1,v1 = hazards(y1,c_idx1,num_reax1,N1,y_tot1,y_tot,beta,gamma,sigma,max_size)     
        
        ############################### pole 2 ##################################
        dummy = y2[0]
        y2[0]=1
        c_idx2 = np.where(y2>0)[0]                                  
        y2[0]=dummy
        N2 = len(c_idx2)
        num_reax2 = N2+1+int(((N2-1)**2-N2+1)/2)+2*(N2-1)  
        a2,v2 = hazards(y2,c_idx2,num_reax2,N2,y_tot2,y_tot,beta,gamma,sigma,max_size)

        a = np.append(a1,a2,axis=0)
        a = np.append(a,[rho*y1[0],rho*y2[0]])
        sum_a=np.sum(a)
        cumsum_a=np.cumsum(a,axis=0)
        epsilon=np.random.uniform(0,1,2)
        j=np.min(np.where(epsilon[0]*sum_a<cumsum_a)) 
        t_f=(np.log(1/epsilon[1]))/sum_a
                     
        if t_y+t_f<tau:  
            t_y+=t_f 
            if (j==num_reax1+num_reax2):
                y2[0]+=1
                y1[0]-=1
            elif (j==num_reax1+num_reax2+1):
                y2[0]-=1
                y1[0]+=1
            elif (num_reax1<=j and j<num_reax1+num_reax2):
                y2+=v2[j-num_reax1,:]  
            else:
                y1+=v1[j,:]    
                
            y1,y2,y_tot1,y_tot2,y_tot = find_tot_plasmids(y1,y2,max_size)
            if (j==0 or j==num_reax1):
                x= np.append(x,y_tot)
                t_x =np.append(t_x,[t_y])
        else:
            t_y=tau
            x=np.append(x,x[-1])
            t_x =np.append(t_x,[t_y])  
    y = np.append(y1,y2,axis=0)
    return y,x,t_x

#@jit
def sim_cells(current,beta,gamma,sigma,max_size):
    y0 = current.init_clusters
    y0[1]=y0[0]
    x0 = np.dot(np.arange(1,max_size),y0[1:max_size])
    y0[1]=0 
    if x0>0:
        yy,xx,tt = ssa(y0,x0,current.div-current.birth,beta,gamma,sigma,max_size)
    else:
        yy = y0
        xx = np.array([0,0])
        tt = np.array([0,current.div-current.birth])
    return yy,xx,tt

def pop_sim(Q,tau,beta,gamma,sigma,max_size,Nmax):
    cells =[]  #processed cells
    N=0  #cell index
    keep = 1500  #keep number of living cells
    time_stamp = (np.log2(keep)+1)*60 +30 # approximate time "keep" number of cells are produced  
    while len(cells)<=Nmax:
        current = Q[0]
        yy,xx,tt = sim_cells(current,beta,gamma,sigma,max_size)
        #update current cell
        current.fin_clusters = yy
        current.x = xx
        current.t = tt+current.birth   
        cells.extend([current])  
            
        if len(Q)>=keep:
            rem_cells = random.sample(Q[1:],1)  #cells to be removed
            #rem_sim = rem_cells[0]
            #yr,xr,tr = sim_cells(rem_sim,beta,gamma,sigma,max_size)
            #rem_sim.fin_clusters = yr
            #rem_sim.x = xr
            #rem_sim.t = tr+rem_sim.birth   
            #cells.extend([rem_sim])  
            new_Q = [x for x in Q if x not in rem_cells]
            Q = new_Q.copy()
        
        Q.remove(current)  #remove current cell from Q
        
        #generate daughter cells  [index,init_clusters,parent,birth,tau]
        daughter1 = cell(N+1,np.append(yy[:max_size],np.zeros(max_size)),current.index,current.t[-1],tau)
        daughter2 = cell(N+2,np.append(yy[max_size:],np.zeros(max_size)),current.index,current.t[-1],tau)
        N+=2
        Q.extend([daughter1,daughter2])  #add daugthers to Q
        Q = sorted(Q, key=lambda x: x.div, reverse=False) #sort Q wrt cell birth time
        
        if len(cells)%201==0:
            print('cells',len(cells),'Q',len(Q))  
    return cells,Q

In [ ]:
y0 = np.zeros(2*max_size)
y0[0]=x0

c= cell(0,y0,[],0,tau)   #index,clusters,parent,birth
Q=[c]

In [ ]:
cells, Q = pop_sim(Q,tau,beta,gamma,sigma,max_size,Nmax)
#with open("test", "wb") as fp:   #Pickling
#    pickle.dump(cells, fp)

#with open("test", "rb") as fp:   # Unpickling
#    load_cells = pickle.load(fp)

In [ ]:
# distribution of number of clusters 
time_stamp =500
gg = pop_check(cells,time_stamp)
thresh = 7
cluster_num = np.zeros(len(gg))
for i in range(len(gg)):
    yy = gg[i].fin_clusters
    cluster_num[i] = np.sum(yy[thresh:max_size])+np.sum(yy[max_size+thresh:])

height = np.bincount(cluster_num.astype(int))/len(cluster_num)
print(height)
dummy=np.sum(height[4:])
height[4]=dummy
heights = np.delete(height,np.arange(5,len(height)))
bars = plt.bar(np.arange(len(heights))-0.2,heights,width=0.4,color='C0',alpha=0.6,label='Simulated') 
plt.text(-0.25,0.85,'Number of Cells: %d' %len(cluster_num))
plt.bar(np.arange(5)+0.2,[0.04,0.18,0.48,0.24,0.06],width=0.4,color='C1',alpha=0.6,label='Experimental Data')
plt.ylim([0,1])
plt.xticks(range(len(heights)))
plt.xlabel('Number of Clusters of size at least %d plasmids' %(thresh))
plt.ylabel('PMF')
plt.legend()
plt.show()


In [ ]:
################ inheritance ratio #######################

time_stamps = [400,500,600,650]

f=plt.figure()
for k in range(len(time_stamps)):
    gg = pop_check(cells,time_stamps[k])
    inheritance = np.zeros(len(gg))
    for i in range(len(gg)):
        parent = [x for x in cells if x.index == gg[i].parent]
        inheritance[i] =gg[i].x[0]/ parent[0].x[-1]
    sns.kdeplot(inheritance,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)))
    #plt.hist(inheritance,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)))
plt.xlabel('Inheritance ratio ($N_d/N_p$)')
plt.ylabel('PDF')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
################ inheritance vs parent plasmid #######################

time_stamps = [400,500,600,650]

f=plt.figure()
for k in range(len(time_stamps)):
    gg = pop_check(cells,time_stamps[k])
    inherited = np.zeros(len(gg))
    parent_copy = np.zeros(len(gg))
    for i in range(len(gg)):
        parent = [x for x in cells if x.index == gg[i].parent]
        inherited[i] = gg[i].x[0]
        parent_copy[i]=parent[0].x[-1]
    #plt.scatter(inherited,parent_copy,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)),s=10,alpha=0.5)
    plt.scatter(inherited,parent_copy-inherited,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)),s=10,alpha=0.5)
plt.title(r'$\rho=$ %1.2f' %(rho))
#plt.xlabel('Number of Inherited Plasmid ($N_d$)')
#plt.ylabel('Parent Plasmid Copy Number ($N_p$)')
plt.xlabel('Daughter Cell 1 Inherited Plasmid ($N_{d_1}$)')
plt.ylabel('Daughter Cell 2 Inherited Plasmid ($N_{d_2}$)')
plt.legend(fontsize=8,loc='upper right')
plt.tight_layout()
plt.axis('square')
plt.xlim([-5,105])
plt.ylim([-5,105])


In [ ]:
################ plasmid copy number #######################
################ instantaneous #######################
time_stamps = [400,500,600,650]
f=plt.figure()
for k in range(len(time_stamps)):
    gg = pop_check(cells,time_stamps[k])
    plasmid_count = np.zeros(len(gg))
    for i in range(len(gg)):
        idx = np.max(np.where(gg[i].t<time_stamps[k]))
        plasmid_count[i]=gg[i].x[idx]
    sns.kdeplot(plasmid_count,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)))
plt.title(r'$\rho=$ %1.2f' %(rho))
plt.xlabel('Plasmid copy number')
plt.legend(fontsize=8,loc='upper left')
plt.ylabel('PDF')

In [ ]:
################ plasmid copy number #######################
################ detected at end of cell cycle #######################

col=['r','C1','C2']
time_stamps = [400,500,600,650]
f=plt.figure(figsize=(4,4))
for k in range(len(time_stamps)):
    gg = pop_check(cells,time_stamps[k])
    plasmid_count = np.zeros(len(gg))
    for i in range(len(gg)):
        plasmid_count[i]=gg[i].x[-1]
    #sns.kdeplot(plasmid_count,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)))
    sns.distplot(plasmid_count,bins=15,kde = False,norm_hist=True,label='%d minutes (%d cells)' %(time_stamps[k],len(gg)))
    print(np.mean(plasmid_count))
plt.title(r'$\rho=$ %1.2f' %(rho))
plt.xlabel('Plasmid copy number')
plt.ylabel('PDF')
plt.legend(fontsize=8,loc='upper right')
plt.xlim([-5,105])
plt.ylim([0,0.14])
plt.yticks([0,0.07,0.14])

In [ ]:
# distribution of plasmid-free cellls
time_stamps = [400,500,600,650]
f=plt.figure()
for k in range(len(time_stamps)):
    gg = pop_check(cells,time_stamps[k])
    hh = [x for x in gg if x.x[-1] == 0]
    #print(len(hh)/len(gg))
    plt.scatter([time_stamps[k]],[len(hh)/len(gg)],s=30,c='C0')
plt.title(r'$\rho=$ %1.2f' %(rho))
plt.xlabel(r'Time ($t$)')
plt.ylabel('Percentage of plasmid-free cells')
plt.xticks(np.linspace(200,4400,5))
#plt.legend(fontsize=9)
#plt.xlim([-5,105])

In [ ]:
########## inheritance daughter 1 vs daughter 2
gg = pop_check(cells,time_stamp)
inherited = np.zeros(len(gg))
parent_copy = np.zeros(len(gg))
for i in range(len(gg)):
    parent = [x for x in cells if x.index == gg[i].parent]
    inherited[i] = gg[i].x[0]
    parent_copy[i]=parent[0].x[-1]
plt.scatter(inherited,parent_copy-inherited,label='%d minutes (%d cells)' %(time_stamp,len(gg)),s=10,alpha=0.5)
plt.title(r'$\rho=$ %1.2f' %(rho))
plt.xlabel('Daughter Cell 1 Inherited Plasmid ($N_{d_1}$)')
plt.ylabel('Daughter Cell 2 Inherited Plasmid ($N_{d_2}$)')
plt.legend(fontsize=8,loc='upper right')
plt.tight_layout()
plt.axis('square')
plt.xlim([-5,105])
plt.ylim([-5,105])
